# Step 1: Defining class to store the State of the warehouse

In [47]:
class Warehouse:
  def __init__(self,rows,columns,board,current_player,depth,A_score=0,B_score=0,remaining_energy_cells=None):
    self.r = rows
    self.c = columns
    self.board = [list(row) for row in board]
    self.current_player = current_player
    self.depth = depth
    self.A_score = A_score
    self.B_score = B_score

    # Always find A_pos and B_pos by scanning the board to ensure consistency
    self.A_pos = None
    self.B_pos = None
    for r_idx in range(self.r):
        for c_idx in range(self.c):
            if self.board[r_idx][c_idx] == 'A':
                self.A_pos = (r_idx, c_idx)
            elif self.board[r_idx][c_idx] == 'B':
                self.B_pos = (r_idx, c_idx)

    # Initialize remaining_energy_cells
    if remaining_energy_cells is None:
        self.remaining_energy_cells = []
        for r_idx in range(self.r):
            for c_idx in range(self.c):
                if self.board[r_idx][c_idx] == 'E':
                    self.remaining_energy_cells.append((r_idx, c_idx))
    else:
        self.remaining_energy_cells = list(remaining_energy_cells)

  def print(self):
    print('+'+('---+'*self.c))
    for i in range(self.r):
      print('| '+' | '.join(self.board[i])+' |')
      print('+'+('---+'*self.c))
  def state(self):
    return (self.A_pos, self.B_pos, len(self.remaining_energy_cells), self.current_player, (self.A_score, self.B_score))

# Step 2: Defining Useful Functions

In [48]:
nodes_generated_minimax = 0
nodes_expanded_minimax = 0
nodes_pruned_minimax = 0

def manhattan_distance(pos1, pos2):
    return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

def get_valid_moves(row, col, other_robot_pos, rows, cols, board):
    moves = []
    possible_moves = {
        'UP': (-1, 0),
        'RIGHT': (0, 1),
        'DOWN': (1, 0),
        'LEFT': (0, -1)
    }

    for direction, (dr, dc) in possible_moves.items():
        new_row, new_col = row + dr, col + dc
        if (0 <= new_row < rows and 0 <= new_col < cols and
            board[new_row][new_col] != '#' and
            (new_row, new_col) != other_robot_pos): # Ensure robot does not move onto other robot's position
            moves.append((direction, (new_row, new_col)))
    return moves

def evaluate_state(warehouse):
    max_score = warehouse.A_score
    min_score = warehouse.B_score

    score_difference = max_score - min_score

    A_pos = warehouse.A_pos
    B_pos = warehouse.B_pos
    board = warehouse.board
    rows = warehouse.r
    cols = warehouse.c

    energy_positions = warehouse.remaining_energy_cells

    positional_advantage = 0
    if not energy_positions:
        positional_advantage = 0
    else:
        min_dist_A_to_E = float('inf')
        for e_pos in energy_positions:
            min_dist_A_to_E = min(min_dist_A_to_E, manhattan_distance(A_pos, e_pos))
        min_dist_B_to_E = float('inf')
        for e_pos in energy_positions:
            min_dist_B_to_E = min(min_dist_B_to_E, manhattan_distance(B_pos, e_pos))

        positional_advantage = min_dist_B_to_E - min_dist_A_to_E

    return score_difference + positional_advantage

def generate_successors(warehouse_state):
    successors = []
    current_board = warehouse_state.board
    current_player = warehouse_state.current_player
    rows, cols = warehouse_state.r, warehouse_state.c

    A_pos_current = warehouse_state.A_pos
    B_pos_current = warehouse_state.B_pos

    robot_pos_moving = A_pos_current if current_player == 'MAX' else B_pos_current
    robot_char_moving = 'A' if current_player == 'MAX' else 'B'
    other_robot_pos = B_pos_current if current_player == 'MAX' else A_pos_current

    valid_moves = get_valid_moves(robot_pos_moving[0], robot_pos_moving[1], other_robot_pos, rows, cols, current_board)

    for move_direction, new_robot_pos_moving in valid_moves:
        new_board = [list(row) for row in current_board]

        new_A_score = warehouse_state.A_score
        new_B_score = warehouse_state.B_score
        new_remaining_energy_cells = list(warehouse_state.remaining_energy_cells)

        # Clear the old position of the moving robot
        new_board[robot_pos_moving[0]][robot_pos_moving[1]] = '.'

        # Check if the new position is an energy cell
        if new_board[new_robot_pos_moving[0]][new_robot_pos_moving[1]] == 'E':
            if current_player == 'MAX':
                new_A_score += 10
            else: # MIN
                new_B_score += 10
            new_board[new_robot_pos_moving[0]][new_robot_pos_moving[1]] = robot_char_moving
            if new_robot_pos_moving in new_remaining_energy_cells:
                new_remaining_energy_cells.remove(new_robot_pos_moving)
        else: # Moving to an empty cell ('.')
            new_board[new_robot_pos_moving[0]][new_robot_pos_moving[1]] = robot_char_moving

        next_player = 'MIN' if current_player == 'MAX' else 'MAX'

        new_warehouse = Warehouse(
            rows, cols, new_board, next_player, warehouse_state.depth,
            A_score=new_A_score, B_score=new_B_score, remaining_energy_cells=new_remaining_energy_cells
        )

        successors.append((move_direction, new_warehouse))

    return successors

def Heuristic_Move_order(warehouse_state,valid_moves):
    # Moves collecting an energy cell first
    ordered_moves = []
    for move_direction, new_robot_pos_moving in valid_moves:
        best_manhattan_distance = float('inf')
        for energy_position in warehouse_state.remaining_energy_cells:
            if manhattan_distance(new_robot_pos_moving, energy_position) < best_manhattan_distance:
                best_manhattan_distance = manhattan_distance(new_robot_pos_moving, energy_position)
        ordered_moves.append((move_direction, best_manhattan_distance))
    ordered_moves.sort(key=lambda x: x[1])
    return [move[0] for move in ordered_moves]

# Step 3: Implementing DIfferent Algorithms

### Part A: Minimax without Heuristics and without Alpha-Beta pruning

In [49]:
def _minimax_pure_recursive(warehouse_state, current_depth, max_depth):
    global nodes_generated_minimax, nodes_expanded_minimax

    if current_depth == max_depth or not warehouse_state.remaining_energy_cells:
        return evaluate_state(warehouse_state)

    nodes_expanded_minimax += 1

    current_board = warehouse_state.board
    current_player = warehouse_state.current_player
    rows, cols = warehouse_state.r, warehouse_state.c

    A_pos_current = warehouse_state.A_pos
    B_pos_current = warehouse_state.B_pos

    robot_pos_moving = A_pos_current if current_player == 'MAX' else B_pos_current
    other_robot_pos = B_pos_current if current_player == 'MAX' else A_pos_current

    valid_moves_for_robot = get_valid_moves(robot_pos_moving[0], robot_pos_moving[1], other_robot_pos, rows, cols, current_board)

    all_successors_map = {move: state for move, state in generate_successors(warehouse_state)}

    ordered_move_directions = list(all_successors_map.keys()) # Default order

    if warehouse_state.current_player == 'MAX':
        max_eval = -float('inf')
        for move_dir in ordered_move_directions:
            if move_dir in all_successors_map:
                successor_state = all_successors_map[move_dir]
                nodes_generated_minimax += 1
                eval = _minimax_pure_recursive(successor_state, current_depth + 1, max_depth)
                max_eval = max(max_eval, eval)
        return max_eval
    else: # MIN player
        min_eval = float('inf')
        for move_dir in ordered_move_directions:
            if move_dir in all_successors_map:
                successor_state = all_successors_map[move_dir]
                nodes_generated_minimax += 1
                eval = _minimax_pure_recursive(successor_state, current_depth + 1, max_depth)
                min_eval = min(min_eval, eval)
        return min_eval

def find_best_move_minimax_pure(initial_warehouse_state):
    global nodes_generated_minimax, nodes_expanded_minimax
    nodes_generated_minimax = 0
    nodes_expanded_minimax = 0
    best_move = None

    if initial_warehouse_state.current_player == 'MAX':
        best_eval = -float('inf')
    else: # MIN
        best_eval = float('inf')

    current_board = initial_warehouse_state.board
    current_player = initial_warehouse_state.current_player
    rows, cols = initial_warehouse_state.r, initial_warehouse_state.c
    A_pos_current = initial_warehouse_state.A_pos
    B_pos_current = initial_warehouse_state.B_pos
    robot_pos_moving = A_pos_current if current_player == 'MAX' else B_pos_current
    other_robot_pos = B_pos_current if current_player == 'MAX' else A_pos_current
    valid_moves_for_robot = get_valid_moves(robot_pos_moving[0], robot_pos_moving[1], other_robot_pos, rows, cols, current_board)

    initial_successors_map = {move: state for move, state in generate_successors(initial_warehouse_state)}
    ordered_move_directions = list(initial_successors_map.keys()) # Default order for initial moves

    nodes_expanded_minimax += 1

    for move_dir in ordered_move_directions:
        if move_dir in initial_successors_map:
            successor_state = initial_successors_map[move_dir]
            nodes_generated_minimax += 1
            eval_for_this_move = _minimax_pure_recursive(successor_state, 1, initial_warehouse_state.depth)

            if initial_warehouse_state.current_player == 'MAX':
                if eval_for_this_move > best_eval:
                    best_eval = eval_for_this_move
                    best_move = move_dir
            else: # MIN player
                if eval_for_this_move < best_eval:
                    best_eval = eval_for_this_move
                    best_move = move_dir

    return best_move, best_eval, nodes_generated_minimax, nodes_expanded_minimax



### Part B: Minimax with Alpha Beta pruning but without Heuristics

In [50]:
def _minimax_alphabeta_recursive(warehouse_state, current_depth, max_depth, alpha, beta):
    global nodes_generated_minimax, nodes_expanded_minimax, nodes_pruned_minimax

    if current_depth == max_depth or not warehouse_state.remaining_energy_cells:
        return evaluate_state(warehouse_state)

    nodes_expanded_minimax += 1

    current_board = warehouse_state.board
    current_player = warehouse_state.current_player
    rows, cols = warehouse_state.r, warehouse_state.c

    A_pos_current = warehouse_state.A_pos
    B_pos_current = warehouse_state.B_pos

    robot_pos_moving = A_pos_current if current_player == 'MAX' else B_pos_current
    other_robot_pos = B_pos_current if current_player == 'MAX' else A_pos_current

    valid_moves_for_robot = get_valid_moves(robot_pos_moving[0], robot_pos_moving[1], other_robot_pos, rows, cols, current_board)

    all_successors_map = {move: state for move, state in generate_successors(warehouse_state)}

    ordered_move_directions = list(all_successors_map.keys()) # Default order

    if warehouse_state.current_player == 'MAX':
        max_eval = -float('inf')
        for move_dir in ordered_move_directions:
            if move_dir in all_successors_map:
                successor_state = all_successors_map[move_dir]
                nodes_generated_minimax += 1
                eval = _minimax_alphabeta_recursive(successor_state, current_depth + 1, max_depth, alpha, beta)
                max_eval = max(max_eval, eval)
                alpha = max(alpha, eval)
                if beta <= alpha:
                    nodes_pruned_minimax += 1 # Increment pruned nodes
                    break # Prune
        return max_eval
    else: # MIN player
        min_eval = float('inf')
        for move_dir in ordered_move_directions:
            if move_dir in all_successors_map:
                successor_state = all_successors_map[move_dir]
                nodes_generated_minimax += 1
                eval = _minimax_alphabeta_recursive(successor_state, current_depth + 1, max_depth, alpha, beta)
                min_eval = min(min_eval, eval)
                beta = min(beta, eval)
                if beta <= alpha:
                    nodes_pruned_minimax += 1 # Increment pruned nodes
                    break # Prune
        return min_eval

def find_best_move_minimax_alphabeta(initial_warehouse_state):
    global nodes_generated_minimax, nodes_expanded_minimax, nodes_pruned_minimax
    nodes_generated_minimax = 0
    nodes_expanded_minimax = 0
    nodes_pruned_minimax = 0 # Reset pruned nodes
    best_move = None

    alpha = -float('inf')
    beta = float('inf')

    if initial_warehouse_state.current_player == 'MAX':
        best_eval = -float('inf')
    else: # MIN
        best_eval = float('inf')

    current_board = initial_warehouse_state.board
    current_player = initial_warehouse_state.current_player
    rows, cols = initial_warehouse_state.r, initial_warehouse_state.c
    A_pos_current = initial_warehouse_state.A_pos
    B_pos_current = initial_warehouse_state.B_pos
    robot_pos_moving = A_pos_current if current_player == 'MAX' else B_pos_current
    other_robot_pos = B_pos_current if current_player == 'MAX' else A_pos_current
    valid_moves_for_robot = get_valid_moves(robot_pos_moving[0], robot_pos_moving[1], other_robot_pos, rows, cols, current_board)

    initial_successors_map = {move: state for move, state in generate_successors(initial_warehouse_state)}
    ordered_move_directions = list(initial_successors_map.keys())  # Normal UP→RIGHT→DOWN→LEFT order for initial moves

    nodes_expanded_minimax += 1 # This counts the initial state expansion

    for move_dir in ordered_move_directions:
        if move_dir in initial_successors_map:
            successor_state = initial_successors_map[move_dir]
            nodes_generated_minimax += 1
            eval_for_this_move = _minimax_alphabeta_recursive(successor_state, 1, initial_warehouse_state.depth, alpha, beta)

            if initial_warehouse_state.current_player == 'MAX':
                if eval_for_this_move > best_eval:
                    best_eval = eval_for_this_move
                    best_move = move_dir
                alpha = max(alpha, eval_for_this_move)
            else: # MIN player
                if eval_for_this_move < best_eval:
                    best_eval = eval_for_this_move
                    best_move = move_dir
                beta = min(beta, eval_for_this_move)

    return best_move, best_eval, nodes_generated_minimax, nodes_expanded_minimax

### Part C: Minimax with Heuristics but without Alpha Beta pruning

In [51]:
def _minimax_heuristic_recursive(warehouse_state, current_depth, max_depth):
    global nodes_generated_minimax, nodes_expanded_minimax

    if current_depth == max_depth or not warehouse_state.remaining_energy_cells:
        return evaluate_state(warehouse_state)

    nodes_expanded_minimax += 1

    current_board = warehouse_state.board
    current_player = warehouse_state.current_player
    rows, cols = warehouse_state.r, warehouse_state.c

    A_pos_current = warehouse_state.A_pos
    B_pos_current = warehouse_state.B_pos

    robot_pos_moving = A_pos_current if current_player == 'MAX' else B_pos_current
    other_robot_pos = B_pos_current if current_player == 'MAX' else A_pos_current

    valid_moves_for_robot = get_valid_moves(robot_pos_moving[0], robot_pos_moving[1], other_robot_pos, rows, cols, current_board)

    all_successors_map = {move: state for move, state in generate_successors(warehouse_state)}

    ordered_move_directions = Heuristic_Move_order(warehouse_state, valid_moves_for_robot)


    if warehouse_state.current_player == 'MAX':
        max_eval = -float('inf')
        for move_dir in ordered_move_directions:
            if move_dir in all_successors_map:
                successor_state = all_successors_map[move_dir]
                nodes_generated_minimax += 1
                eval = _minimax_heuristic_recursive(successor_state, current_depth + 1, max_depth)
                max_eval = max(max_eval, eval)
        return max_eval
    else: # MIN player
        min_eval = float('inf')
        for move_dir in ordered_move_directions:
            if move_dir in all_successors_map:
                successor_state = all_successors_map[move_dir]
                nodes_generated_minimax += 1
                eval = _minimax_heuristic_recursive(successor_state, current_depth + 1, max_depth)
                min_eval = min(min_eval, eval)
        return min_eval

def find_best_move_minimax_heuristic(initial_warehouse_state):
    global nodes_generated_minimax, nodes_expanded_minimax
    nodes_generated_minimax = 0
    nodes_expanded_minimax = 0
    best_move = None

    if initial_warehouse_state.current_player == 'MAX':
        best_eval = -float('inf')
    else: # MIN
        best_eval = float('inf')

    current_board = initial_warehouse_state.board
    current_player = initial_warehouse_state.current_player
    rows, cols = initial_warehouse_state.r, initial_warehouse_state.c
    A_pos_current = initial_warehouse_state.A_pos
    B_pos_current = initial_warehouse_state.B_pos
    robot_pos_moving = A_pos_current if current_player == 'MAX' else B_pos_current
    other_robot_pos = B_pos_current if current_player == 'MAX' else A_pos_current
    valid_moves_for_robot = get_valid_moves(robot_pos_moving[0], robot_pos_moving[1], other_robot_pos, rows, cols, current_board)

    initial_successors_map = {move: state for move, state in generate_successors(initial_warehouse_state)}
    ordered_move_directions = list(initial_successors_map.keys()) # Default order for initial moves

    nodes_expanded_minimax += 1

    for move_dir in ordered_move_directions:
        if move_dir in initial_successors_map:
            successor_state = initial_successors_map[move_dir]
            nodes_generated_minimax += 1
            eval_for_this_move = _minimax_heuristic_recursive(successor_state, 1, initial_warehouse_state.depth)

            if initial_warehouse_state.current_player == 'MAX':
                if eval_for_this_move > best_eval:
                    best_eval = eval_for_this_move
                    best_move = move_dir
            else: # MIN player
                if eval_for_this_move < best_eval:
                    best_eval = eval_for_this_move
                    best_move = move_dir

    return best_move, best_eval, nodes_generated_minimax, nodes_expanded_minimax

### Part D: Minimax with Alpha Beta pruning and with Heuristics

In [52]:
def _minimax_alphabeta_heuristic_recursive(warehouse_state, current_depth, max_depth, alpha, beta):
    global nodes_generated_minimax, nodes_expanded_minimax, nodes_pruned_minimax

    if current_depth == max_depth or not warehouse_state.remaining_energy_cells:
        return evaluate_state(warehouse_state)

    nodes_expanded_minimax += 1

    current_board = warehouse_state.board
    current_player = warehouse_state.current_player
    rows, cols = warehouse_state.r, warehouse_state.c

    A_pos_current = warehouse_state.A_pos
    B_pos_current = warehouse_state.B_pos

    robot_pos_moving = A_pos_current if current_player == 'MAX' else B_pos_current
    other_robot_pos = B_pos_current if current_player == 'MAX' else A_pos_current

    valid_moves_for_robot = get_valid_moves(robot_pos_moving[0], robot_pos_moving[1], other_robot_pos, rows, cols, current_board)

    all_successors_map = {move: state for move, state in generate_successors(warehouse_state)}

    ordered_move_directions = Heuristic_Move_order(warehouse_state, valid_moves_for_robot)

    if warehouse_state.current_player == 'MAX':
        max_eval = -float('inf')
        for move_dir in ordered_move_directions:
            if move_dir in all_successors_map:
                successor_state = all_successors_map[move_dir]
                nodes_generated_minimax += 1
                eval = _minimax_alphabeta_heuristic_recursive(successor_state, current_depth + 1, max_depth, alpha, beta)
                max_eval = max(max_eval, eval)
                alpha = max(alpha, eval)
                if beta <= alpha:
                    nodes_pruned_minimax += 1 # Increment pruned nodes
                    break # Prune
        return max_eval
    else: # MIN player
        min_eval = float('inf')
        for move_dir in ordered_move_directions:
            if move_dir in all_successors_map:
                successor_state = all_successors_map[move_dir]
                nodes_generated_minimax += 1
                eval = _minimax_alphabeta_heuristic_recursive(successor_state, current_depth + 1, max_depth, alpha, beta)
                min_eval = min(min_eval, eval)
                beta = min(beta, eval)
                if beta <= alpha:
                    nodes_pruned_minimax += 1 # Increment pruned nodes
                    break # Prune
        return min_eval

def find_best_move_minimax_alphabeta_heuristic(initial_warehouse_state):
    global nodes_generated_minimax, nodes_expanded_minimax, nodes_pruned_minimax
    nodes_generated_minimax = 0
    nodes_expanded_minimax = 0
    nodes_pruned_minimax = 0 # Reset pruned nodes
    best_move = None

    alpha = -float('inf')
    beta = float('inf')

    if initial_warehouse_state.current_player == 'MAX':
        best_eval = -float('inf')
    else: # MIN
        best_eval = float('inf')

    current_board = initial_warehouse_state.board
    current_player = initial_warehouse_state.current_player
    rows, cols = initial_warehouse_state.r, initial_warehouse_state.c
    A_pos_current = initial_warehouse_state.A_pos
    B_pos_current = initial_warehouse_state.B_pos
    robot_pos_moving = A_pos_current if current_player == 'MAX' else B_pos_current
    other_robot_pos = B_pos_current if current_player == 'MAX' else A_pos_current
    valid_moves_for_robot = get_valid_moves(robot_pos_moving[0], robot_pos_moving[1], other_robot_pos, rows, cols, current_board)

    initial_successors_map = {move: state for move, state in generate_successors(initial_warehouse_state)}
    ordered_move_directions = Heuristic_Move_order(initial_warehouse_state, valid_moves_for_robot) # Heuristic order for initial moves

    nodes_expanded_minimax += 1 # This counts the initial state expansion

    for move_dir in ordered_move_directions:
        if move_dir in initial_successors_map:
            successor_state = initial_successors_map[move_dir]
            nodes_generated_minimax += 1
            eval_for_this_move = _minimax_alphabeta_heuristic_recursive(successor_state, 1, initial_warehouse_state.depth, alpha, beta)

            if initial_warehouse_state.current_player == 'MAX':
                if eval_for_this_move > best_eval:
                    best_eval = eval_for_this_move
                    best_move = move_dir
                alpha = max(alpha, eval_for_this_move)
            else: # MIN player
                if eval_for_this_move < best_eval:
                    best_eval = eval_for_this_move
                    best_move = move_dir
                beta = min(beta, eval_for_this_move)

    return best_move, best_eval, nodes_generated_minimax, nodes_expanded_minimax

# Execution: Sample test case

In [53]:
r,c=map(int,input("Enter R and C separated by space: ").split())
print('Enter the board:')
board = []
for i in range(r):
  board.append(list(input()))
current_player = input('Enter the starting player (MAX/MIN): ')
depth = int(input('Enter the max depth for heuristic calculation: '))
Bot_Warehouse = Warehouse(r,c,board,current_player,depth)
Bot_Warehouse.print()

Enter R and C separated by space: 5 7
Enter the board:
A.E.#..
.#..#E.
...#...
#E#..#.
....E.B
Enter the starting player (MAX/MIN): MAX
Enter the max depth for heuristic calculation: 5
+---+---+---+---+---+---+---+
| A | . | E | . | # | . | . |
+---+---+---+---+---+---+---+
| . | # | . | . | # | E | . |
+---+---+---+---+---+---+---+
| . | . | . | # | . | . | . |
+---+---+---+---+---+---+---+
| # | E | # | . | . | # | . |
+---+---+---+---+---+---+---+
| . | . | . | . | E | . | B |
+---+---+---+---+---+---+---+


# Execution: Execution of all the algorithms

In [56]:
import pandas as pd
import time

comparison_results = {}

# --- Minimax (pure, no heuristic ordering, no pruning) ---
start_time = time.time()
best_move, eval_score, nodes_gen, nodes_exp = find_best_move_minimax_pure(Bot_Warehouse)
execution_time = time.time() - start_time

print("Algorithm: Minimax (Normal Move Order)")
print(f"Best Move: {best_move}")
print(f"Evaluation Score = {eval_score}")
print(f"Nodes Generated = {nodes_gen}")
print(f"Nodes Expanded = {nodes_exp}")
print(f"Search Depth = {Bot_Warehouse.depth}")
print(f"Execution Time = {execution_time:.6f}s")

comparison_results['Minimax'] = {
    'Best Move': best_move,
    'Evaluation Score': eval_score,
    'Nodes Generated': nodes_gen,
    'Nodes Expanded': nodes_exp,
    'Nodes Pruned': 0,
    'Search Depth': Bot_Warehouse.depth,
    'Total Time (s)': execution_time
}

# --- Minimax with Heuristic move ordering (no pruning) ---
start_time = time.time()
best_move, eval_score, nodes_gen, nodes_exp = find_best_move_minimax_heuristic(Bot_Warehouse)
execution_time = time.time() - start_time

print("Algorithm: Minimax (Heuristic Move Order)")
print(f"Best Move: {best_move}")
print(f"Evaluation Score = {eval_score}")
print(f"Nodes Generated = {nodes_gen}")
print(f"Nodes Expanded = {nodes_exp}")
print(f"Search Depth = {Bot_Warehouse.depth}")
print(f"Execution Time = {execution_time:.6f}s")

comparison_results['Minimax + Heuristic Order'] = {
    'Best Move': best_move,
    'Evaluation Score': eval_score,
    'Nodes Generated': nodes_gen,
    'Nodes Expanded': nodes_exp,
    'Nodes Pruned': 0,
    'Search Depth': Bot_Warehouse.depth,
    'Total Time (s)': execution_time
}

# --- Alpha-Beta, Case 1: normal move order (UP -> RIGHT -> DOWN -> LEFT) ---
start_time = time.time()
best_move, eval_score, nodes_gen, nodes_exp = find_best_move_minimax_alphabeta(Bot_Warehouse)
execution_time = time.time() - start_time

print("Algorithm: Alpha-Beta (Normal Move Order)")
print(f"Best Move: {best_move}")
print(f"Evaluation Score = {eval_score}")
print(f"Nodes Generated = {nodes_gen}")
print(f"Nodes Expanded = {nodes_exp}")
print(f"Nodes Pruned = {nodes_pruned_minimax}")
print(f"Search Depth = {Bot_Warehouse.depth}")
print(f"Execution Time = {execution_time:.6f}s")

comparison_results['Alpha-Beta (Normal Order)'] = {
    'Best Move': best_move,
    'Evaluation Score': eval_score,
    'Nodes Generated': nodes_gen,
    'Nodes Expanded': nodes_exp,
    'Nodes Pruned': nodes_pruned_minimax,
    'Search Depth': Bot_Warehouse.depth,
    'Total Time (s)': execution_time
}

# --- Alpha-Beta, Case 2: heuristic move order (energy-collecting moves first) ---
start_time = time.time()
best_move, eval_score, nodes_gen, nodes_exp = find_best_move_minimax_alphabeta_heuristic(Bot_Warehouse)
execution_time = time.time() - start_time

print("Algorithm: Alpha-Beta (Heuristic Move Order)")
print(f"Best Move: {best_move}")
print(f"Evaluation Score = {eval_score}")
print(f"Nodes Generated = {nodes_gen}")
print(f"Nodes Expanded = {nodes_exp}")
print(f"Nodes Pruned = {nodes_pruned_minimax}")
print(f"Search Depth = {Bot_Warehouse.depth}")
print(f"Execution Time = {execution_time:.6f}s")

comparison_results['Alpha-Beta (Heuristic Order)'] = {
    'Best Move': best_move,
    'Evaluation Score': eval_score,
    'Nodes Generated': nodes_gen,
    'Nodes Expanded': nodes_exp,
    'Nodes Pruned': nodes_pruned_minimax,
    'Search Depth': Bot_Warehouse.depth,
    'Total Time (s)': execution_time
}

df_comparison_revised = pd.DataFrame.from_dict(comparison_results, orient='index')
display(df_comparison_revised[['Best Move', 'Evaluation Score', 'Nodes Generated', 'Nodes Expanded', 'Nodes Pruned', 'Total Time (s)']].round(6))

Algorithm: Minimax (Normal Move Order)
Best Move: RIGHT
Evaluation Score = 1
Nodes Generated = 66
Nodes Expanded = 31
Search Depth = 5
Execution Time = 0.001391s
Algorithm: Minimax (Heuristic Move Order)
Best Move: RIGHT
Evaluation Score = 1
Nodes Generated = 66
Nodes Expanded = 31
Search Depth = 5
Execution Time = 0.001276s
Algorithm: Alpha-Beta (Normal Move Order)
Best Move: RIGHT
Evaluation Score = 1
Nodes Generated = 35
Nodes Expanded = 20
Nodes Pruned = 7
Search Depth = 5
Execution Time = 0.000671s
Algorithm: Alpha-Beta (Heuristic Move Order)
Best Move: RIGHT
Evaluation Score = 1
Nodes Generated = 29
Nodes Expanded = 18
Nodes Pruned = 8
Search Depth = 5
Execution Time = 0.000770s


,Best Move,Evaluation Score,Nodes Generated,Nodes Expanded,Nodes Pruned,Total Time (s)
Minimax,RIGHT,1,66,31,0,0.001391
Minimax + Heuristic Order,RIGHT,1,66,31,0,0.001276
Alpha-Beta (Normal Order),RIGHT,1,35,20,7,0.000671
Alpha-Beta (Heuristic Order),RIGHT,1,29,18,8,0.000770
